# Projet A2 · Scoring crédit : accorder ou refuser un prêt · ⭐⭐⭐

**Piste « Projets avancés » · Niveau ⭐⭐⭐ Avancé · Pipeline complet façon soutenance**

- **Question métier** : une banque en ligne reçoit des demandes de prêt personnel. Pour chaque dossier, faut-il **accorder** ou **refuser** ? Un défaut de paiement coûte bien plus cher qu'un bon client refusé — le modèle doit en tenir compte.
- **Ce qu'on construit** : un pipeline complet — nettoyage, exploration, traitement du **déséquilibre** (22 % de défauts), 4 modèles candidats, **XGBoost réglé par Optuna**, **seuil de décision optimisé sur un coût métier**, interprétation **SHAP** et une **fiche « décision expliquée »** pour chaque client.
- **Livrable** : ce notebook exécuté + un rapport de 10-15 slides selon `../gabarit-rapport.md` + une phrase de synthèse.

Comment l'utiliser :
- Google Colab (aucun GPU nécessaire) ou en local dans l'environnement `projets-avances/requirements.txt`. `Maj + Entrée` cellule après cellule.
- `MODE_RAPIDE = True` (par défaut) : 2 000 dossiers et 20 essais Optuna, tout tourne en 2-3 minutes sur CPU. `MODE_RAPIDE = False` : les 32 000 dossiers et 60 essais (≈ 10-15 min sur Colab).
- Les cellules **« À toi »** sont des exercices : le squelette s'exécute tel quel, la vérification affiche ✅ / ❌, la solution est repliée juste en dessous. Essaie avant d'ouvrir !
- Les cellules **« Rapport »** impriment les chiffres à recopier dans ton rapport.

## 0. Préparation

In [ ]:
# Colab n'a ni Optuna ni SHAP par défaut.
# On n'installe que ce qui manque : la cellule ne fait rien si tout est déjà là.
import importlib.util, subprocess, sys

manquants = [paquet for module, paquet in [("xgboost", "xgboost"), ("shap", "shap"), ("optuna", "optuna"), ("imblearn", "imbalanced-learn")]
             if importlib.util.find_spec(module) is None]
if manquants:
    print("installation :", ", ".join(manquants))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *manquants], check=True)
else:
    print("Tout est déjà installé.")


In [ ]:
import os, io, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODE_RAPIDE = True          # False = jeu complet + recherche Optuna longue
SEED = 42
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_columns", 30)

try:                        # sur certains Mac, Python ne trouve pas les certificats HTTPS sans ce réglage
    import certifi
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
except ImportError:
    pass

DATA_DIR = Path("data")     # cache local des données (dossier ignoré par git)
DATA_DIR.mkdir(exist_ok=True)
print("MODE_RAPIDE =", MODE_RAPIDE)

In [ ]:
resultats = {}

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = un booléen ou une fonction)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else " — pas encore, relis l'énoncé ou ouvre l'indice."))

def proche(a, b, tol=0.05):
    """Vrai si a est à moins de tol (en relatif) de b."""
    return abs(a - b) <= tol * max(abs(b), 1e-9)

print("Helpers prêts.")

## 1. Contexte et question métier

Une banque en ligne reçoit chaque jour des demandes de prêt personnel (études, santé, travaux, création d'entreprise…). Pour chaque dossier elle connaît le demandeur (revenu, situation de logement, ancienneté dans l'emploi, historique de crédit) et le prêt demandé (montant, taux, objet, note interne de A à G). Elle veut un **score de risque** : la probabilité que le client ne rembourse pas (`loan_status = 1`), et surtout une **règle de décision** : au-dessus de quel score refuse-t-on ?

À qui ça sert ? Aux analystes crédit, qui traitent aujourd'hui les dossiers à la main, et à la direction des risques, qui doit prouver au régulateur que la décision est **explicable** — un client refusé a le droit de savoir pourquoi. C'est le cadre du projet OpenClassrooms « P7 » : un modèle de scoring, un seuil, et un tableau de bord qui explique chaque décision.

**Métrique choisie** : 22 % de défauts seulement — la « justesse » (accuracy) est un piège (refuser tout le monde = 78 %). On suit le **ROC-AUC** (la capacité à classer les défauts au-dessus des bons clients, indépendante du seuil) et le **PR-AUC** (plus sensible sur la classe rare). Puis on choisit le seuil sur un **coût métier** : un défaut non détecté (faux négatif) coûte le capital prêté, un bon client refusé (faux positif) coûte une marge perdue — on posera **FN = 10 × FP**. Un bon résultat = un coût total nettement inférieur à celui du seuil naïf 0,5, avec un modèle explicable dossier par dossier.

## 2. Les données

Source : [Credit Risk Dataset sur OpenML (id 43454)](https://www.openml.org/d/43454), licence CC0, 32 581 dossiers × 12 colonnes (copie du [dataset Kaggle de Lao Tse](https://www.kaggle.com/datasets/laotse/credit-risk-dataset)). Chargement : `fetch_openml`, sinon un miroir GitHub, sinon un échantillon de 500 dossiers intégré à la cellule suivante. Le fichier est mis en cache dans `data/`.

### Dictionnaire des variables

| Variable | Sens | Valeurs |
|---|---|---|
| `person_age` | âge du demandeur | années |
| `person_income` | revenu annuel | dollars |
| `person_home_ownership` | logement | `RENT`, `OWN`, `MORTGAGE`, `OTHER` |
| `person_emp_length` | ancienneté dans l'emploi | années (manquant pour 3 %) |
| `loan_intent` | objet du prêt | `EDUCATION`, `MEDICAL`, `VENTURE`, `PERSONAL`, `DEBTCONSOLIDATION`, `HOMEIMPROVEMENT` |
| `loan_grade` | note interne du prêt | `A` (sûr) → `G` (risqué) |
| `loan_amnt` | montant demandé | dollars |
| `loan_int_rate` | taux d'intérêt | % (manquant pour 10 %) |
| **`loan_status`** | **cible** : 1 = défaut de paiement, 0 = remboursé | 0 / 1 |
| `loan_percent_income` | mensualités / revenu | ratio |
| `cb_person_default_on_file` | défaut déjà enregistré au bureau de crédit | `Y` / `N` |
| `cb_person_cred_hist_length` | longueur de l'historique de crédit | années |

In [ ]:
DONNEES_SECOURS = """person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
24,41300,RENT,2.0,EDUCATION,C,2400,12.68,0,0.06,N,2
24,85000,RENT,5.0,MEDICAL,B,25000,10.62,0,0.29,N,4
24,88000,MORTGAGE,4.0,MEDICAL,B,6000,9.91,0,0.07,N,2
23,81120,MORTGAGE,3.0,PERSONAL,B,15000,10.25,0,0.18,N,4
39,73000,MORTGAGE,3.0,VENTURE,A,10000,7.51,0,0.14,N,14
29,33864,RENT,2.0,VENTURE,D,6000,14.91,1,0.18,Y,6
34,35000,RENT,2.0,VENTURE,B,15000,10.99,1,0.43,N,5
28,54000,RENT,2.0,EDUCATION,D,10000,16.49,1,0.19,Y,7
23,64000,RENT,0.0,MEDICAL,C,12000,13.16,0,0.19,Y,4
22,85000,MORTGAGE,6.0,VENTURE,C,10000,13.99,0,0.12,Y,4
29,76176,MORTGAGE,12.0,MEDICAL,A,4800,6.99,0,0.06,N,7
23,24996,OWN,1.0,EDUCATION,B,8000,11.26,0,0.32,N,2
27,75000,RENT,2.0,DEBTCONSOLIDATION,C,15000,14.35,0,0.2,N,5
24,80000,OWN,0.0,VENTURE,A,5500,8.94,0,0.07,N,2
21,25500,RENT,6.0,EDUCATION,C,2000,13.47,0,0.08,N,2
23,78000,MORTGAGE,1.0,DEBTCONSOLIDATION,C,8000,12.87,1,0.1,Y,3
25,85000,RENT,5.0,DEBTCONSOLIDATION,B,12000,12.42,0,0.14,N,4
23,40000,MORTGAGE,7.0,MEDICAL,A,4000,7.51,0,0.1,N,2
36,60000,RENT,1.0,PERSONAL,C,7200,,0,0.12,Y,12
24,48000,MORTGAGE,4.0,VENTURE,A,5000,5.99,0,0.1,N,4
27,70000,MORTGAGE,9.0,EDUCATION,A,16000,7.9,0,0.23,N,7
26,40000,OWN,0.0,MEDICAL,B,15000,9.64,0,0.38,N,4
30,31000,MORTGAGE,3.0,VENTURE,D,12000,14.42,1,0.39,N,9
27,33000,RENT,11.0,HOMEIMPROVEMENT,D,1600,16.89,1,0.05,Y,5
26,85000,RENT,2.0,VENTURE,A,28000,7.49,1,0.33,N,3
26,49000,RENT,2.0,VENTURE,D,8400,18.25,1,0.17,Y,4
26,98400,MORTGAGE,4.0,VENTURE,A,8450,6.99,0,0.09,N,2
29,100000,MORTGAGE,7.0,MEDICAL,C,15000,12.04,0,0.15,N,5
22,50508,RENT,5.0,VENTURE,A,6000,5.42,0,0.12,N,3
24,19200,RENT,2.0,EDUCATION,B,1000,12.53,0,0.05,N,4
48,29120,RENT,0.0,DEBTCONSOLIDATION,D,3000,12.49,1,0.1,Y,14
25,95000,RENT,4.0,EDUCATION,A,24000,8.9,0,0.25,N,2
22,45000,MORTGAGE,3.0,EDUCATION,B,4000,11.83,0,0.09,N,3
22,110000,RENT,7.0,EDUCATION,C,20000,14.17,0,0.18,N,3
21,70000,RENT,3.0,PERSONAL,C,3300,,0,0.05,N,4
29,71000,RENT,4.0,DEBTCONSOLIDATION,A,9700,8.94,0,0.14,N,5
27,80000,MORTGAGE,3.0,DEBTCONSOLIDATION,B,3000,11.11,0,0.04,N,7
47,75000,OTHER,1.0,DEBTCONSOLIDATION,C,12000,12.84,0,0.16,N,11
22,40000,OWN,4.0,EDUCATION,A,8000,5.79,0,0.2,N,2
23,16800,RENT,2.0,MEDICAL,D,2750,16.77,1,0.16,Y,2
22,62004,MORTGAGE,6.0,EDUCATION,A,3600,7.37,0,0.06,N,2
22,52000,RENT,6.0,EDUCATION,C,18000,,1,0.35,N,4
31,115000,MORTGAGE,4.0,VENTURE,C,6500,13.98,0,0.06,N,9
34,70000,RENT,0.0,DEBTCONSOLIDATION,B,5000,9.91,0,0.07,N,7
26,45000,RENT,,DEBTCONSOLIDATION,B,1400,10.59,0,0.03,N,4
22,56000,RENT,1.0,EDUCATION,A,12000,6.91,0,0.21,N,2
27,110000,MORTGAGE,4.0,PERSONAL,A,2500,8.07,0,0.02,N,6
23,105000,MORTGAGE,7.0,VENTURE,B,12000,11.36,0,0.11,N,2
33,33600,OWN,3.0,VENTURE,C,9500,12.99,0,0.28,N,8
30,114000,MORTGAGE,13.0,DEBTCONSOLIDATION,A,9975,5.42,0,0.09,N,10
27,49000,MORTGAGE,11.0,MEDICAL,B,8500,10.59,0,0.17,N,6
24,15915,RENT,0.0,PERSONAL,D,2000,15.05,1,0.13,Y,4
22,55000,RENT,2.0,EDUCATION,A,6000,8.9,0,0.11,N,3
32,50000,RENT,0.0,EDUCATION,C,6400,13.35,0,0.13,N,8
22,54000,RENT,0.0,DEBTCONSOLIDATION,C,7000,13.61,0,0.13,N,4
27,75000,MORTGAGE,4.0,PERSONAL,B,20150,11.36,0,0.27,N,6
34,39000,RENT,5.0,HOMEIMPROVEMENT,B,7200,11.71,1,0.18,N,5
32,60000,RENT,3.0,HOMEIMPROVEMENT,B,4000,10.99,0,0.07,N,10
28,18000,RENT,9.0,HOMEIMPROVEMENT,A,7600,8.49,1,0.42,N,9
25,370000,MORTGAGE,5.0,DEBTCONSOLIDATION,C,15000,12.99,0,0.04,Y,2
22,78792,MORTGAGE,4.0,EDUCATION,D,5500,14.61,0,0.07,N,4
22,30000,RENT,3.0,EDUCATION,C,4100,14.26,0,0.14,Y,3
30,55000,MORTGAGE,9.0,HOMEIMPROVEMENT,B,16000,,0,0.29,N,6
30,39600,MORTGAGE,5.0,DEBTCONSOLIDATION,B,3000,10.38,0,0.08,N,10
32,42000,RENT,0.0,EDUCATION,D,2000,15.62,1,0.05,N,6
30,24000,RENT,0.0,DEBTCONSOLIDATION,B,5000,10.59,0,0.21,N,6
29,80000,MORTGAGE,13.0,EDUCATION,A,10000,7.9,0,0.13,N,9
24,72000,MORTGAGE,0.0,EDUCATION,B,9000,,0,0.13,N,2
34,58650,MORTGAGE,7.0,MEDICAL,G,12000,20.17,1,0.17,Y,10
31,42000,RENT,6.0,EDUCATION,D,1200,16.29,0,0.03,Y,5
31,39996,MORTGAGE,3.0,VENTURE,B,9600,11.26,0,0.24,N,7
32,113000,MORTGAGE,1.0,EDUCATION,A,5000,,0,0.04,N,6
29,44000,RENT,1.0,EDUCATION,D,4000,12.36,1,0.09,Y,8
25,94000,MORTGAGE,1.0,VENTURE,E,25000,,1,0.27,Y,2
27,26400,RENT,0.0,PERSONAL,B,8000,10.59,1,0.3,N,7
32,45000,MORTGAGE,6.0,MEDICAL,A,6400,7.49,0,0.14,N,9
47,79500,MORTGAGE,0.0,MEDICAL,A,16000,7.51,0,0.2,N,14
38,40000,RENT,19.0,VENTURE,B,12000,10.59,0,0.3,N,11
23,60000,MORTGAGE,6.0,PERSONAL,B,2000,10.36,0,0.03,N,2
21,38400,RENT,5.0,EDUCATION,B,5000,10.59,0,0.13,N,3
23,115000,RENT,0.0,PERSONAL,C,10800,14.65,0,0.09,N,4
21,15000,RENT,0.0,PERSONAL,D,3600,15.95,1,0.24,N,2
25,175000,MORTGAGE,10.0,HOMEIMPROVEMENT,A,8000,7.9,0,0.05,N,2
26,65000,MORTGAGE,10.0,HOMEIMPROVEMENT,A,8000,6.99,0,0.12,N,3
47,37000,MORTGAGE,6.0,PERSONAL,A,7400,8.0,0,0.2,N,13
39,52000,RENT,5.0,PERSONAL,B,6500,,0,0.13,N,12
38,125000,RENT,0.0,EDUCATION,A,12000,7.66,0,0.1,N,14
34,35000,RENT,0.0,MEDICAL,B,3200,11.26,0,0.09,N,6
30,58000,RENT,3.0,PERSONAL,C,20000,14.72,1,0.34,N,5
22,78000,MORTGAGE,6.0,EDUCATION,E,16000,16.45,0,0.21,Y,2
26,42000,RENT,4.0,DEBTCONSOLIDATION,B,2250,10.38,0,0.05,N,4
22,53124,MORTGAGE,4.0,EDUCATION,C,6500,13.61,0,0.12,N,3
24,88000,MORTGAGE,0.0,HOMEIMPROVEMENT,C,6400,11.66,1,0.07,Y,3
30,143000,MORTGAGE,11.0,PERSONAL,A,18000,8.49,0,0.13,N,7
22,75000,MORTGAGE,2.0,MEDICAL,A,8150,7.51,0,0.11,N,4
29,420000,MORTGAGE,5.0,DEBTCONSOLIDATION,C,8000,12.73,0,0.02,N,5
26,53000,OWN,3.0,VENTURE,B,6000,9.99,0,0.11,N,2
26,70000,MORTGAGE,0.0,EDUCATION,A,7000,7.88,0,0.1,N,4
27,85000,MORTGAGE,11.0,EDUCATION,A,12000,6.62,0,0.14,N,10
24,79800,RENT,8.0,HOMEIMPROVEMENT,B,2500,9.62,0,0.03,N,2
30,54000,RENT,1.0,EDUCATION,E,16000,17.99,1,0.3,Y,10
26,52500,MORTGAGE,0.0,PERSONAL,B,15250,9.63,0,0.29,N,2
22,40499,MORTGAGE,7.0,MEDICAL,B,11500,11.83,0,0.28,N,4
25,58000,MORTGAGE,9.0,MEDICAL,B,10400,12.18,0,0.18,N,2
35,51000,RENT,8.0,DEBTCONSOLIDATION,C,8000,13.11,0,0.16,Y,10
23,30000,OWN,,VENTURE,A,7000,7.49,0,0.23,N,3
32,42000,RENT,5.0,VENTURE,C,3500,14.35,0,0.08,Y,7
46,60000,RENT,5.0,MEDICAL,B,4800,12.12,0,0.08,N,16
35,88000,MORTGAGE,4.0,PERSONAL,D,10000,14.11,1,0.11,Y,6
22,12240,RENT,1.0,PERSONAL,D,1000,14.96,1,0.08,Y,3
23,33000,RENT,3.0,DEBTCONSOLIDATION,B,4000,10.96,0,0.12,N,4
26,38000,RENT,3.0,EDUCATION,B,13400,9.99,1,0.35,N,2
24,60000,RENT,8.0,EDUCATION,A,16800,6.54,0,0.28,N,2
23,77000,MORTGAGE,7.0,EDUCATION,A,10000,7.88,0,0.13,N,4
28,92000,MORTGAGE,12.0,VENTURE,F,24500,18.3,0,0.27,N,7
25,61000,OWN,2.0,PERSONAL,C,15000,14.65,0,0.25,N,2
41,26400,MORTGAGE,4.0,DEBTCONSOLIDATION,B,6000,12.21,1,0.23,N,16
29,120000,RENT,3.0,DEBTCONSOLIDATION,A,15000,9.63,0,0.13,N,6
26,33600,RENT,4.0,PERSONAL,D,3300,15.65,0,0.1,Y,2
27,60000,OWN,4.0,HOMEIMPROVEMENT,A,10500,7.49,0,0.17,N,7
23,93600,RENT,3.0,VENTURE,D,25000,15.21,0,0.27,N,4
21,39996,MORTGAGE,5.0,PERSONAL,B,8000,10.25,0,0.2,N,4
28,94000,RENT,3.0,EDUCATION,A,3700,7.9,0,0.04,N,9
32,48750,RENT,5.0,DEBTCONSOLIDATION,C,3000,13.49,0,0.06,N,7
22,29554,MORTGAGE,,DEBTCONSOLIDATION,A,5775,5.42,1,0.2,N,3
21,15874,RENT,,EDUCATION,B,1500,,0,0.09,N,3
38,30120,RENT,1.0,HOMEIMPROVEMENT,D,14400,12.8,1,0.48,Y,17
42,40800,MORTGAGE,5.0,DEBTCONSOLIDATION,A,10400,7.51,0,0.25,N,14
28,73500,RENT,0.0,HOMEIMPROVEMENT,C,14400,12.98,0,0.2,Y,6
38,140000,MORTGAGE,1.0,PERSONAL,D,10000,14.09,0,0.07,N,16
28,60000,MORTGAGE,5.0,MEDICAL,B,6000,10.99,0,0.1,N,6
23,48000,RENT,7.0,DEBTCONSOLIDATION,C,3500,13.98,0,0.07,N,4
25,85000,MORTGAGE,6.0,DEBTCONSOLIDATION,C,1500,13.06,0,0.02,N,2
38,120000,RENT,3.0,DEBTCONSOLIDATION,C,10000,10.28,0,0.08,Y,12
23,120000,MORTGAGE,7.0,DEBTCONSOLIDATION,A,4900,7.14,0,0.04,N,4
25,75000,MORTGAGE,9.0,MEDICAL,C,4800,15.27,0,0.06,Y,3
21,42000,OTHER,5.0,DEBTCONSOLIDATION,C,12000,14.26,0,0.29,Y,4
21,27000,RENT,1.0,EDUCATION,A,4800,7.51,0,0.18,N,2
29,53360,MORTGAGE,11.0,EDUCATION,A,10475,7.88,0,0.2,N,9
27,59004,RENT,3.0,EDUCATION,D,10000,16.32,0,0.17,Y,6
33,38500,RENT,1.0,DEBTCONSOLIDATION,A,5000,7.14,0,0.13,N,10
44,62400,RENT,,HOMEIMPROVEMENT,E,10000,16.82,1,0.16,N,15
23,97600,MORTGAGE,7.0,MEDICAL,A,14500,7.51,0,0.15,N,2
26,83957,MORTGAGE,4.0,MEDICAL,A,3000,8.94,0,0.04,N,3
23,120000,RENT,7.0,EDUCATION,B,12000,11.71,0,0.1,N,4
24,30000,RENT,8.0,EDUCATION,A,7500,5.79,0,0.25,N,2
26,56004,MORTGAGE,10.0,EDUCATION,C,20000,13.57,0,0.36,Y,4
42,85000,MORTGAGE,10.0,HOMEIMPROVEMENT,D,8500,15.58,0,0.1,Y,12
27,54000,RENT,11.0,VENTURE,D,6000,14.59,0,0.11,Y,5
30,49200,MORTGAGE,13.0,HOMEIMPROVEMENT,A,9250,,0,0.19,N,8
36,60000,RENT,0.0,HOMEIMPROVEMENT,C,15000,13.57,0,0.25,N,12
33,48000,OWN,5.0,VENTURE,B,14400,12.42,0,0.3,N,10
30,44760,RENT,1.0,EDUCATION,B,3300,11.71,0,0.07,N,8
43,65000,OWN,4.0,MEDICAL,B,15000,11.71,0,0.23,N,13
26,71000,MORTGAGE,2.0,DEBTCONSOLIDATION,A,4400,5.42,0,0.06,N,4
23,55000,MORTGAGE,7.0,DEBTCONSOLIDATION,A,4000,5.42,0,0.07,N,3
27,120000,MORTGAGE,0.0,PERSONAL,A,7000,7.49,0,0.06,N,9
32,75000,RENT,3.0,VENTURE,D,12800,14.74,0,0.17,Y,6
22,70000,MORTGAGE,5.0,DEBTCONSOLIDATION,C,8000,13.49,0,0.11,Y,2
30,36000,OWN,5.0,VENTURE,A,4800,,0,0.13,N,6
26,200000,MORTGAGE,10.0,HOMEIMPROVEMENT,B,12000,11.71,0,0.06,N,2
30,30000,RENT,2.0,PERSONAL,C,8000,13.22,0,0.27,Y,9
43,55000,RENT,3.0,DEBTCONSOLIDATION,B,11750,10.99,0,0.21,N,16
25,40000,RENT,6.0,MEDICAL,B,10000,10.0,0,0.25,N,4
25,100000,RENT,3.0,DEBTCONSOLIDATION,A,2600,7.9,0,0.03,N,3
30,198000,MORTGAGE,5.0,DEBTCONSOLIDATION,E,15000,,1,0.08,Y,9
25,50950,RENT,5.0,HOMEIMPROVEMENT,B,20000,11.48,1,0.39,N,3
24,57772,RENT,6.0,VENTURE,C,13500,13.99,0,0.23,N,2
24,53000,RENT,5.0,EDUCATION,A,5100,7.9,0,0.1,N,2
23,25200,RENT,1.0,EDUCATION,A,5500,7.9,0,0.22,N,2
35,55000,MORTGAGE,7.0,EDUCATION,B,1450,10.59,0,0.03,N,5
27,38000,MORTGAGE,3.0,VENTURE,B,4500,9.99,0,0.12,N,7
33,25000,RENT,,DEBTCONSOLIDATION,C,8500,13.43,1,0.34,N,6
29,100000,MORTGAGE,4.0,VENTURE,C,20000,13.57,0,0.2,Y,9
27,62400,MORTGAGE,12.0,MEDICAL,A,12000,5.42,0,0.19,N,6
42,82500,RENT,3.0,DEBTCONSOLIDATION,A,2575,6.76,0,0.03,N,11
28,103992,MORTGAGE,3.0,PERSONAL,B,18000,10.99,0,0.17,N,7
23,78000,RENT,0.0,MEDICAL,B,6000,9.99,0,0.08,N,2
29,29004,RENT,8.0,EDUCATION,C,6000,12.87,0,0.21,N,5
33,68000,MORTGAGE,15.0,VENTURE,B,5000,12.18,0,0.07,N,10
43,36000,MORTGAGE,,HOMEIMPROVEMENT,A,4000,6.62,1,0.11,N,16
38,55000,RENT,2.0,PERSONAL,B,14000,11.48,0,0.25,N,14
23,67200,MORTGAGE,7.0,VENTURE,A,16000,6.03,0,0.24,N,2
25,22000,RENT,3.0,PERSONAL,E,5000,16.32,1,0.23,N,3
27,64000,OWN,11.0,VENTURE,A,5000,5.79,0,0.08,N,5
29,125000,MORTGAGE,2.0,HOMEIMPROVEMENT,C,8500,13.47,0,0.07,Y,5
27,130000,OWN,3.0,MEDICAL,A,15000,6.03,0,0.12,N,10
28,55000,MORTGAGE,5.0,PERSONAL,A,4800,5.79,0,0.09,N,6
29,28550,RENT,6.0,DEBTCONSOLIDATION,B,4000,10.59,0,0.14,N,10
32,46000,RENT,16.0,PERSONAL,B,10000,10.59,0,0.22,N,7
22,12996,OWN,1.0,VENTURE,B,4750,,1,0.37,N,2
35,40000,OWN,0.0,PERSONAL,A,5000,6.92,0,0.13,N,5
32,33000,MORTGAGE,2.0,MEDICAL,A,5000,6.17,0,0.15,N,7
24,75000,MORTGAGE,0.0,EDUCATION,B,7000,11.99,0,0.09,N,3
24,54417,MORTGAGE,1.0,MEDICAL,C,9525,11.66,1,0.18,Y,3
33,33504,RENT,4.0,DEBTCONSOLIDATION,B,9600,10.99,0,0.29,N,8
24,118000,MORTGAGE,4.0,VENTURE,A,28000,7.9,0,0.24,N,4
33,44000,RENT,5.0,VENTURE,D,8000,17.49,0,0.18,N,8
31,80000,RENT,3.0,VENTURE,A,10000,7.66,0,0.13,N,7
28,142000,MORTGAGE,3.0,EDUCATION,B,18000,10.38,0,0.13,N,8
36,24000,RENT,3.0,MEDICAL,A,6500,7.29,0,0.27,N,13
44,210000,MORTGAGE,,VENTURE,B,25000,10.37,0,0.12,N,14
24,110004,MORTGAGE,8.0,DEBTCONSOLIDATION,C,16000,,0,0.15,N,3
23,60000,RENT,3.0,EDUCATION,B,8000,10.25,0,0.13,N,3
28,32400,RENT,3.0,MEDICAL,D,4900,14.12,1,0.15,N,7
23,29120,OWN,0.0,MEDICAL,A,6400,8.94,0,0.22,N,4
21,39000,MORTGAGE,5.0,EDUCATION,A,1000,7.49,0,0.03,N,3
29,92000,RENT,2.0,DEBTCONSOLIDATION,C,20000,13.72,0,0.22,N,10
35,23000,RENT,13.0,MEDICAL,A,6500,8.63,0,0.28,N,6
31,26010,MORTGAGE,2.0,HOMEIMPROVEMENT,B,6500,9.99,0,0.25,N,9
39,115000,MORTGAGE,6.0,VENTURE,D,25000,15.58,0,0.22,Y,17
30,75000,MORTGAGE,3.0,VENTURE,A,18750,7.9,0,0.25,N,6
26,28000,RENT,8.0,EDUCATION,D,6625,12.92,0,0.24,N,2
22,69996,OWN,2.0,MEDICAL,C,8000,12.84,0,0.11,Y,4
35,40000,OWN,5.0,VENTURE,A,8000,7.66,0,0.2,N,6
23,60000,MORTGAGE,3.0,PERSONAL,A,13000,7.88,0,0.22,N,4
25,80000,MORTGAGE,2.0,EDUCATION,B,22000,11.48,0,0.28,N,3
28,42000,MORTGAGE,5.0,HOMEIMPROVEMENT,A,6000,5.42,0,0.14,N,9
22,37000,RENT,2.0,DEBTCONSOLIDATION,E,3000,20.3,1,0.08,N,2
21,26124,RENT,3.0,EDUCATION,D,8875,15.62,1,0.34,Y,3
39,40000,RENT,5.0,EDUCATION,D,6000,14.54,0,0.15,N,11
23,50000,OWN,5.0,VENTURE,A,7000,8.9,0,0.14,N,4
32,89000,RENT,0.0,MEDICAL,A,12000,8.94,0,0.13,N,6
30,96000,RENT,1.0,VENTURE,B,4000,10.75,0,0.04,N,5
38,24000,RENT,5.0,MEDICAL,B,7200,10.65,0,0.3,N,14
23,30000,RENT,,HOMEIMPROVEMENT,C,5000,12.99,1,0.17,Y,2
23,26976,RENT,4.0,PERSONAL,A,9000,7.51,1,0.33,N,4
25,48659,RENT,4.0,EDUCATION,B,7500,,0,0.15,N,4
44,39684,RENT,4.0,PERSONAL,B,6000,,0,0.15,N,15
25,48996,MORTGAGE,8.0,MEDICAL,B,5000,9.88,1,0.1,N,2
29,32000,MORTGAGE,5.0,MEDICAL,A,3100,7.51,1,0.1,N,5
28,84000,RENT,3.0,HOMEIMPROVEMENT,B,15000,11.86,0,0.18,N,9
27,40000,RENT,4.0,DEBTCONSOLIDATION,B,4800,11.71,0,0.12,N,8
23,113000,RENT,2.0,DEBTCONSOLIDATION,C,14000,13.49,0,0.12,Y,2
36,19200,RENT,,DEBTCONSOLIDATION,B,1200,9.62,0,0.06,N,14
36,130000,RENT,11.0,EDUCATION,E,20000,,0,0.15,Y,14
36,56900,MORTGAGE,16.0,VENTURE,G,6200,,1,0.11,N,11
24,14400,RENT,0.0,PERSONAL,D,1600,18.25,1,0.11,N,3
21,37232,RENT,3.0,MEDICAL,B,17500,12.53,1,0.47,N,2
31,190000,MORTGAGE,15.0,MEDICAL,C,24000,,0,0.13,N,10
30,37200,OWN,,VENTURE,A,10000,6.03,0,0.27,N,10
34,68000,RENT,5.0,PERSONAL,B,7200,,0,0.11,N,9
41,58650,MORTGAGE,15.0,MEDICAL,B,12000,11.14,1,0.17,N,16
27,150000,MORTGAGE,11.0,VENTURE,B,8000,12.21,0,0.05,N,5
28,70000,MORTGAGE,2.0,VENTURE,A,5600,6.62,0,0.08,N,5
36,64000,MORTGAGE,20.0,DEBTCONSOLIDATION,E,6800,18.39,1,0.11,Y,12
26,32640,RENT,7.0,VENTURE,B,3000,10.75,0,0.09,N,2
27,70000,RENT,4.0,EDUCATION,B,20000,11.48,0,0.29,N,9
21,28692,MORTGAGE,5.0,EDUCATION,B,3000,10.75,0,0.1,N,2
25,78720,MORTGAGE,6.0,MEDICAL,B,10000,9.62,0,0.13,N,2
24,29000,RENT,5.0,MEDICAL,C,6000,15.27,0,0.21,N,2
27,140000,RENT,4.0,EDUCATION,A,20000,7.9,0,0.14,N,6
25,99275,RENT,1.0,PERSONAL,C,10000,14.65,0,0.1,Y,3
34,35000,RENT,1.0,PERSONAL,A,8000,6.17,0,0.23,N,9
23,70000,MORTGAGE,1.0,VENTURE,B,10000,10.99,0,0.14,N,4
28,225000,MORTGAGE,11.0,EDUCATION,E,9000,16.95,1,0.04,Y,8
29,72000,RENT,4.0,EDUCATION,A,24000,7.66,1,0.33,N,6
31,48000,RENT,1.0,MEDICAL,A,6000,6.03,0,0.13,N,6
21,15600,OWN,5.0,VENTURE,A,1000,7.29,0,0.06,N,4
31,95004,MORTGAGE,3.0,PERSONAL,D,25000,16.07,0,0.26,Y,8
35,200000,MORTGAGE,0.0,HOMEIMPROVEMENT,B,18000,10.99,0,0.09,N,10
34,113000,RENT,6.0,EDUCATION,C,9600,,0,0.08,Y,9
24,29000,MORTGAGE,1.0,HOMEIMPROVEMENT,B,20000,10.62,0,0.69,N,3
37,46680,MORTGAGE,6.0,VENTURE,D,8000,15.21,0,0.17,N,15
22,72000,MORTGAGE,5.0,VENTURE,D,12000,14.84,0,0.17,Y,4
37,55000,OWN,2.0,VENTURE,B,24375,10.37,0,0.44,N,13
27,24500,RENT,3.0,MEDICAL,B,10000,10.25,1,0.41,N,7
23,102000,MORTGAGE,7.0,DEBTCONSOLIDATION,C,6000,,0,0.06,Y,3
31,42000,RENT,1.0,MEDICAL,B,12000,10.99,0,0.29,N,9
28,43000,RENT,12.0,MEDICAL,A,7000,,0,0.16,N,8
44,68000,RENT,7.0,EDUCATION,A,5150,7.9,0,0.08,N,14
24,35000,MORTGAGE,7.0,EDUCATION,A,15000,7.49,0,0.43,N,4
33,114400,MORTGAGE,3.0,PERSONAL,C,20000,13.48,0,0.17,N,5
27,74400,MORTGAGE,11.0,DEBTCONSOLIDATION,D,10000,16.0,1,0.13,Y,6
26,62000,MORTGAGE,1.0,DEBTCONSOLIDATION,A,8400,6.62,0,0.14,N,4
27,104000,RENT,8.0,DEBTCONSOLIDATION,B,11000,,0,0.11,N,8
21,35004,MORTGAGE,1.0,VENTURE,A,3600,8.0,0,0.1,N,2
28,95000,MORTGAGE,0.0,DEBTCONSOLIDATION,A,18000,7.88,0,0.19,N,10
29,86000,MORTGAGE,5.0,PERSONAL,C,14000,14.27,0,0.16,N,9
29,55000,MORTGAGE,11.0,MEDICAL,B,4000,10.99,0,0.07,N,6
25,79000,MORTGAGE,9.0,DEBTCONSOLIDATION,A,10000,5.79,0,0.13,N,2
23,99500,MORTGAGE,7.0,EDUCATION,D,25000,15.7,0,0.25,N,2
22,54000,MORTGAGE,1.0,MEDICAL,C,4000,13.98,0,0.07,N,4
25,60000,RENT,6.0,PERSONAL,B,8000,9.63,0,0.13,N,3
26,65600,RENT,5.0,EDUCATION,D,10000,13.55,1,0.15,Y,2
21,12960,RENT,2.0,VENTURE,B,2525,,1,0.19,N,2
29,24960,RENT,13.0,DEBTCONSOLIDATION,A,4000,5.42,0,0.16,N,10
29,32400,RENT,,MEDICAL,A,5000,8.9,0,0.15,N,8
25,84000,RENT,2.0,VENTURE,A,6000,8.94,0,0.07,N,2
24,168288,RENT,5.0,HOMEIMPROVEMENT,B,1800,11.12,0,0.01,N,3
26,48000,MORTGAGE,0.0,DEBTCONSOLIDATION,C,12000,,0,0.25,N,3
27,32000,RENT,0.0,DEBTCONSOLIDATION,B,15000,,1,0.47,N,7
27,85000,RENT,0.0,VENTURE,A,7200,7.68,0,0.08,N,10
24,18000,RENT,0.0,EDUCATION,B,1200,11.89,0,0.07,N,4
39,50400,RENT,4.0,EDUCATION,D,7000,11.86,0,0.14,N,11
21,72000,RENT,1.0,EDUCATION,C,7500,13.22,0,0.1,N,3
25,100000,OWN,4.0,MEDICAL,C,15000,11.28,0,0.15,N,4
33,50000,MORTGAGE,7.0,HOMEIMPROVEMENT,B,10000,9.99,0,0.2,N,7
25,55000,RENT,3.0,DEBTCONSOLIDATION,A,20000,6.91,1,0.36,N,3
22,91800,MORTGAGE,0.0,MEDICAL,D,16800,15.62,1,0.16,Y,4
23,77200,RENT,1.0,EDUCATION,B,3000,9.63,0,0.04,N,2
28,31000,RENT,7.0,MEDICAL,D,3600,14.11,1,0.12,Y,8
22,50004,MORTGAGE,5.0,EDUCATION,D,14000,15.58,0,0.28,N,4
28,80000,RENT,12.0,MEDICAL,A,17000,5.79,0,0.21,N,6
30,24000,RENT,,MEDICAL,C,1000,15.96,0,0.04,Y,7
22,60000,MORTGAGE,4.0,PERSONAL,C,3000,13.16,0,0.05,N,2
35,84000,OWN,4.0,HOMEIMPROVEMENT,A,9000,6.76,0,0.11,N,6
24,63410,MORTGAGE,3.0,MEDICAL,E,20000,17.51,1,0.27,N,4
34,98000,OWN,19.0,EDUCATION,B,16000,10.99,0,0.16,N,5
27,110000,MORTGAGE,8.0,VENTURE,B,30000,11.11,0,0.27,N,6
30,35000,MORTGAGE,,DEBTCONSOLIDATION,B,22750,10.62,0,0.65,N,10
23,41000,RENT,2.0,EDUCATION,A,5000,7.51,0,0.12,N,2
36,30000,RENT,0.0,EDUCATION,B,9000,12.18,0,0.3,N,17
31,35004,RENT,2.0,EDUCATION,C,2700,13.49,0,0.08,N,5
26,30000,RENT,3.0,EDUCATION,C,8000,12.61,0,0.27,Y,4
27,57504,MORTGAGE,11.0,MEDICAL,B,17000,12.18,1,0.3,N,8
26,78000,MORTGAGE,4.0,PERSONAL,B,1750,10.36,0,0.02,N,2
26,39600,RENT,3.0,PERSONAL,B,6000,12.42,1,0.15,N,3
51,260000,MORTGAGE,1.0,PERSONAL,A,14000,7.66,0,0.05,N,26
24,25000,MORTGAGE,2.0,DEBTCONSOLIDATION,A,10000,8.0,0,0.4,N,2
23,27996,OWN,2.0,MEDICAL,B,9250,10.75,0,0.33,N,2
32,120000,MORTGAGE,13.0,EDUCATION,B,2000,,0,0.02,N,10
24,54996,MORTGAGE,8.0,VENTURE,A,5500,5.99,0,0.1,N,4
29,96000,MORTGAGE,4.0,DEBTCONSOLIDATION,C,15000,,0,0.16,N,6
24,80000,RENT,6.0,EDUCATION,A,25000,6.62,1,0.31,N,2
22,40542,MORTGAGE,6.0,VENTURE,A,6250,,0,0.15,N,2
35,82863,RENT,14.0,VENTURE,A,11000,,0,0.13,N,6
28,70900,MORTGAGE,12.0,EDUCATION,D,25000,15.37,0,0.35,Y,6
24,50000,RENT,0.0,DEBTCONSOLIDATION,A,2800,5.79,0,0.06,N,2
26,96000,MORTGAGE,2.0,VENTURE,G,10000,21.14,1,0.1,N,3
36,19200,OWN,0.0,HOMEIMPROVEMENT,C,4500,13.49,1,0.23,Y,14
25,45000,RENT,2.0,PERSONAL,C,11100,13.11,0,0.25,N,2
42,42000,RENT,2.0,PERSONAL,C,4200,13.35,0,0.1,N,17
40,150000,MORTGAGE,5.0,PERSONAL,B,12000,10.37,0,0.08,N,12
22,31200,RENT,6.0,EDUCATION,B,12000,10.39,1,0.38,N,2
28,62000,RENT,0.0,EDUCATION,A,15000,7.9,0,0.24,N,8
22,28500,RENT,1.0,EDUCATION,B,1500,,0,0.05,N,4
21,59000,RENT,1.0,EDUCATION,A,5500,5.42,0,0.09,N,2
28,24000,RENT,4.0,PERSONAL,C,5600,12.68,0,0.23,Y,5
26,90000,RENT,0.0,MEDICAL,A,12000,9.38,0,0.13,N,3
27,44004,RENT,1.0,EDUCATION,D,11750,15.31,1,0.27,Y,8
22,65004,OWN,3.0,VENTURE,B,4000,10.99,0,0.06,N,3
22,40680,RENT,2.0,EDUCATION,B,6000,11.36,1,0.15,N,4
26,110000,MORTGAGE,4.0,VENTURE,B,5000,9.63,0,0.05,N,2
24,62500,MORTGAGE,8.0,DEBTCONSOLIDATION,B,16000,10.38,0,0.26,N,3
27,65000,MORTGAGE,11.0,VENTURE,A,8000,7.49,0,0.12,N,8
25,28800,RENT,6.0,MEDICAL,A,14800,7.49,1,0.51,N,3
26,50000,RENT,1.0,DEBTCONSOLIDATION,B,7150,12.69,0,0.14,N,2
27,33600,RENT,2.0,PERSONAL,D,6000,14.91,1,0.18,Y,5
27,100000,MORTGAGE,1.0,HOMEIMPROVEMENT,A,12000,6.03,0,0.12,N,7
33,69600,MORTGAGE,17.0,PERSONAL,B,16000,11.99,0,0.23,N,8
28,52000,RENT,0.0,HOMEIMPROVEMENT,C,12000,13.35,0,0.23,N,5
24,69600,RENT,2.0,PERSONAL,A,10000,8.38,0,0.14,N,2
25,46000,RENT,5.0,EDUCATION,B,7000,10.25,0,0.15,N,3
29,60000,MORTGAGE,7.0,DEBTCONSOLIDATION,C,15000,12.87,0,0.25,N,5
21,51996,MORTGAGE,1.0,EDUCATION,A,18000,8.49,0,0.35,N,2
43,36000,RENT,1.0,MEDICAL,A,10000,8.94,0,0.28,N,11
25,53000,MORTGAGE,2.0,VENTURE,B,10750,11.14,0,0.2,N,2
33,42000,RENT,0.0,MEDICAL,A,8000,5.99,0,0.19,N,7
22,39000,RENT,4.0,EDUCATION,A,5000,6.99,0,0.13,N,2
25,50004,OWN,1.0,VENTURE,A,10000,7.29,0,0.2,N,3
24,40000,RENT,0.0,DEBTCONSOLIDATION,A,10800,6.92,0,0.27,N,4
34,42240,RENT,3.0,PERSONAL,C,8775,13.06,0,0.21,Y,6
24,54996,OWN,4.0,EDUCATION,B,10000,10.37,0,0.18,N,4
23,55000,RENT,7.0,PERSONAL,C,8200,13.11,0,0.15,Y,3
33,72000,RENT,3.0,MEDICAL,B,12000,11.99,0,0.17,N,5
22,37392,RENT,1.0,EDUCATION,B,12000,10.62,1,0.32,N,2
30,90000,RENT,2.0,EDUCATION,A,6400,6.62,0,0.07,N,7
30,30000,OWN,4.0,PERSONAL,D,8000,15.7,0,0.27,Y,10
35,50532,RENT,8.0,VENTURE,C,5000,13.57,0,0.1,Y,8
32,44500,OWN,3.0,HOMEIMPROVEMENT,A,15000,7.51,0,0.34,N,5
42,316800,MORTGAGE,3.0,PERSONAL,D,19400,,0,0.06,Y,13
29,77000,MORTGAGE,3.0,PERSONAL,A,5000,7.14,0,0.06,N,7
27,78500,RENT,6.0,EDUCATION,A,8000,8.59,0,0.1,N,8
40,35000,RENT,17.0,EDUCATION,A,2400,,0,0.07,N,16
26,55000,RENT,10.0,MEDICAL,B,15000,11.12,0,0.27,N,2
23,48000,MORTGAGE,7.0,EDUCATION,A,14125,,0,0.29,N,3
23,29000,OWN,,MEDICAL,F,10000,18.62,1,0.34,Y,3
27,38000,RENT,12.0,VENTURE,A,7000,7.66,0,0.18,N,6
22,52000,MORTGAGE,2.0,MEDICAL,B,18000,9.88,0,0.35,N,4
27,52000,MORTGAGE,4.0,HOMEIMPROVEMENT,A,8000,6.99,0,0.15,N,8
27,59000,MORTGAGE,3.0,MEDICAL,B,6000,11.83,0,0.1,N,5
23,19200,RENT,0.0,PERSONAL,B,3000,9.99,1,0.16,N,3
22,31500,RENT,1.0,VENTURE,D,6800,14.11,1,0.22,N,2
29,38000,RENT,5.0,EDUCATION,A,1000,7.88,0,0.03,N,9
26,62500,RENT,7.0,VENTURE,C,12000,12.23,0,0.19,N,4
29,50000,RENT,11.0,EDUCATION,B,5000,10.37,0,0.1,N,5
26,35000,RENT,10.0,PERSONAL,D,8000,,0,0.23,N,3
26,62004,RENT,9.0,DEBTCONSOLIDATION,A,24000,7.9,1,0.39,N,3
31,69000,RENT,12.0,HOMEIMPROVEMENT,A,16000,5.99,0,0.23,N,9
27,36000,RENT,5.0,EDUCATION,B,5000,9.63,0,0.14,N,8
30,34000,RENT,0.0,DEBTCONSOLIDATION,C,12000,12.87,1,0.35,N,6
27,42000,OWN,4.0,EDUCATION,C,10000,14.65,0,0.24,N,7
29,102000,MORTGAGE,13.0,MEDICAL,A,6000,7.88,0,0.06,N,9
22,45600,RENT,5.0,VENTURE,D,10000,14.61,0,0.22,N,4
23,66000,MORTGAGE,7.0,VENTURE,A,15000,6.03,0,0.23,N,4
23,69996,RENT,4.0,PERSONAL,C,15000,13.43,0,0.21,Y,4
27,69000,RENT,5.0,VENTURE,E,25000,13.75,1,0.36,Y,6
23,21600,MORTGAGE,7.0,EDUCATION,A,2000,6.17,0,0.09,N,4
21,35000,RENT,5.0,DEBTCONSOLIDATION,B,7200,12.69,0,0.21,N,3
26,105000,MORTGAGE,0.0,PERSONAL,B,5300,12.69,0,0.05,N,4
29,88000,RENT,8.0,DEBTCONSOLIDATION,C,17600,14.26,0,0.2,Y,6
22,55000,RENT,6.0,DEBTCONSOLIDATION,A,6000,6.99,0,0.11,N,3
25,35000,RENT,,MEDICAL,C,6500,12.68,0,0.19,Y,2
47,80004,MORTGAGE,5.0,EDUCATION,B,7000,,0,0.09,N,15
26,66000,MORTGAGE,5.0,MEDICAL,B,15000,9.88,0,0.23,N,4
24,52000,RENT,2.0,DEBTCONSOLIDATION,B,10000,9.91,0,0.19,N,2
25,63000,MORTGAGE,3.0,EDUCATION,C,3000,13.47,0,0.05,N,2
24,61147,MORTGAGE,5.0,MEDICAL,A,17000,6.03,0,0.28,N,4
25,43200,OWN,9.0,VENTURE,B,13800,11.48,0,0.32,N,2
23,57000,MORTGAGE,6.0,EDUCATION,D,17500,14.09,0,0.31,Y,3
22,54000,RENT,4.0,MEDICAL,C,8075,13.06,0,0.15,N,3
31,81000,RENT,8.0,VENTURE,C,7000,13.61,0,0.09,Y,8
24,14400,RENT,0.0,MEDICAL,A,1200,7.29,0,0.08,N,4
25,25000,RENT,1.0,HOMEIMPROVEMENT,B,8000,12.42,1,0.32,N,3
24,58650,RENT,1.0,MEDICAL,D,10000,18.25,1,0.14,N,4
21,57192,MORTGAGE,0.0,MEDICAL,A,8000,7.49,0,0.14,N,3
23,130208,MORTGAGE,7.0,DEBTCONSOLIDATION,C,20000,12.73,0,0.15,N,3
27,27000,OWN,1.0,VENTURE,A,6000,8.0,0,0.22,N,6
23,45000,RENT,2.0,PERSONAL,A,12000,5.79,0,0.27,N,3
22,29570,RENT,6.0,DEBTCONSOLIDATION,D,8000,14.42,1,0.27,Y,4
22,57000,RENT,6.0,EDUCATION,C,12000,13.11,0,0.21,N,2
25,50000,MORTGAGE,4.0,DEBTCONSOLIDATION,A,6000,6.76,0,0.12,N,2
23,113000,RENT,8.0,DEBTCONSOLIDATION,D,35000,18.25,1,0.31,N,4
23,50400,RENT,3.0,VENTURE,B,5000,10.38,0,0.1,N,2
24,109000,MORTGAGE,3.0,DEBTCONSOLIDATION,B,8800,12.69,0,0.08,N,2
25,25716,RENT,1.0,MEDICAL,A,9250,6.54,1,0.36,N,3
44,55000,RENT,18.0,PERSONAL,B,9600,11.26,0,0.17,N,16
28,58200,RENT,11.0,EDUCATION,E,20500,16.82,1,0.35,Y,5
28,50000,RENT,4.0,MEDICAL,A,8000,7.14,0,0.16,N,7
24,131004,MORTGAGE,8.0,PERSONAL,A,24000,7.49,0,0.18,N,4
26,70000,MORTGAGE,4.0,EDUCATION,C,4000,13.98,0,0.06,Y,4
22,80000,RENT,5.0,EDUCATION,C,10000,14.27,0,0.13,N,2
26,48000,RENT,5.0,VENTURE,F,25000,18.43,1,0.52,N,2
31,143000,RENT,4.0,VENTURE,D,25000,16.07,0,0.17,Y,7
24,54000,RENT,1.0,PERSONAL,C,10000,13.35,0,0.19,N,4
25,100000,RENT,9.0,HOMEIMPROVEMENT,C,20000,13.06,0,0.2,N,2
23,52000,RENT,7.0,VENTURE,B,7500,10.38,0,0.14,N,2
22,54000,RENT,4.0,EDUCATION,C,8000,13.48,1,0.15,Y,3
29,25000,RENT,3.0,MEDICAL,B,5000,10.95,0,0.2,N,10
34,45000,OWN,0.0,VENTURE,A,7000,5.42,0,0.16,N,5
22,48000,MORTGAGE,7.0,EDUCATION,B,6500,10.62,0,0.14,N,2
22,71000,MORTGAGE,6.0,VENTURE,C,1500,15.23,0,0.02,Y,2
32,55000,RENT,0.0,DEBTCONSOLIDATION,B,18000,10.38,1,0.33,N,7
37,90000,RENT,21.0,VENTURE,A,10000,8.94,0,0.11,N,12
37,49000,MORTGAGE,3.0,MEDICAL,D,4000,15.65,1,0.08,N,17
25,36400,RENT,2.0,PERSONAL,B,18200,9.91,1,0.5,N,4
28,108000,MORTGAGE,6.0,EDUCATION,B,27400,10.99,0,0.25,N,7
23,30000,RENT,0.0,EDUCATION,B,4800,,0,0.16,N,4
21,78000,RENT,2.0,DEBTCONSOLIDATION,D,10000,15.57,1,0.13,Y,4
22,41504,RENT,6.0,EDUCATION,C,4800,12.99,0,0.12,N,4
23,25200,OWN,3.0,EDUCATION,B,1000,11.48,0,0.04,N,4
27,64000,RENT,3.0,MEDICAL,B,18000,11.11,0,0.28,N,8
30,96000,RENT,3.0,PERSONAL,B,25000,,0,0.26,N,6
26,49493,MORTGAGE,6.0,VENTURE,C,15250,13.11,1,0.31,Y,3
23,35004,RENT,1.0,EDUCATION,B,6000,,0,0.17,N,2
26,72500,RENT,1.0,VENTURE,B,10000,9.96,0,0.14,N,3
42,13200,RENT,4.0,DEBTCONSOLIDATION,C,2500,,1,0.19,Y,14
30,50000,MORTGAGE,0.0,EDUCATION,D,10000,16.89,0,0.2,N,10
27,29120,RENT,2.0,MEDICAL,B,4000,10.75,0,0.14,N,8
24,42000,RENT,0.0,VENTURE,A,12000,7.66,0,0.29,N,4
23,110000,MORTGAGE,1.0,PERSONAL,B,18000,10.37,0,0.16,N,3
41,102000,OWN,5.0,EDUCATION,A,7000,6.92,0,0.07,N,12
24,69000,MORTGAGE,1.0,EDUCATION,B,4900,10.96,1,0.07,N,2
23,65000,MORTGAGE,5.0,DEBTCONSOLIDATION,A,2275,,0,0.04,N,2
25,38000,RENT,1.0,EDUCATION,A,10000,7.49,0,0.26,N,2
24,60000,RENT,4.0,EDUCATION,B,12500,11.49,0,0.21,N,2
41,75000,RENT,15.0,PERSONAL,D,4000,16.29,0,0.05,Y,12
22,85000,RENT,0.0,DEBTCONSOLIDATION,A,10000,7.66,0,0.12,N,3
21,55404,RENT,5.0,DEBTCONSOLIDATION,C,6500,13.98,0,0.12,N,4
33,160000,MORTGAGE,3.0,DEBTCONSOLIDATION,B,22000,10.75,0,0.14,N,9
22,84000,MORTGAGE,0.0,MEDICAL,A,10000,7.88,0,0.12,N,3
21,50000,RENT,1.0,DEBTCONSOLIDATION,A,10000,6.99,0,0.2,N,4
30,75000,MORTGAGE,0.0,HOMEIMPROVEMENT,C,15000,12.98,0,0.2,N,8
26,33227,RENT,2.0,EDUCATION,B,4000,9.88,0,0.12,N,3
25,30000,RENT,2.0,VENTURE,E,5200,17.19,1,0.17,Y,2
25,216000,MORTGAGE,2.0,HOMEIMPROVEMENT,D,25000,16.07,0,0.12,N,2
34,40000,RENT,2.0,EDUCATION,B,5000,12.53,0,0.13,N,5
31,96000,MORTGAGE,15.0,HOMEIMPROVEMENT,D,8000,14.09,0,0.08,Y,9
29,47000,MORTGAGE,1.0,DEBTCONSOLIDATION,B,5000,9.76,0,0.11,N,10
30,21600,RENT,4.0,VENTURE,B,5000,11.71,0,0.23,N,7
22,32000,RENT,1.0,PERSONAL,D,5400,16.29,1,0.17,N,2
31,39000,MORTGAGE,5.0,EDUCATION,B,22000,11.36,1,0.56,N,5
24,54000,RENT,3.0,MEDICAL,C,2400,13.49,1,0.04,Y,4
35,38400,MORTGAGE,1.0,VENTURE,B,9500,10.36,0,0.25,N,6
33,30000,RENT,0.0,DEBTCONSOLIDATION,B,1600,10.25,0,0.05,N,9
27,87000,MORTGAGE,2.0,VENTURE,A,7200,9.63,0,0.08,N,6
21,45996,MORTGAGE,2.0,EDUCATION,C,17000,13.16,1,0.37,Y,4
26,125000,MORTGAGE,5.0,HOMEIMPROVEMENT,B,7400,12.53,0,0.06,N,2
28,80500,MORTGAGE,4.0,DEBTCONSOLIDATION,A,6000,7.74,0,0.07,N,5
30,58000,OWN,7.0,PERSONAL,A,6000,5.79,0,0.1,N,5
29,60000,MORTGAGE,,PERSONAL,C,14125,13.49,0,0.24,N,5
22,42000,MORTGAGE,,PERSONAL,A,3200,7.9,1,0.08,N,3
27,36000,RENT,11.0,MEDICAL,A,3000,8.0,0,0.08,N,9
24,47000,MORTGAGE,7.0,MEDICAL,B,15000,10.74,0,0.32,N,2
23,120000,MORTGAGE,7.0,EDUCATION,C,8000,13.61,0,0.07,Y,2
25,45000,RENT,4.0,MEDICAL,B,5100,10.0,0,0.11,N,4
23,78000,RENT,5.0,VENTURE,D,3000,16.89,0,0.04,N,3
46,48000,MORTGAGE,0.0,PERSONAL,A,9600,,0,0.2,N,17
29,24000,RENT,6.0,PERSONAL,B,5000,10.74,0,0.21,N,10
"""

In [ ]:
URL_MIROIR = "https://raw.githubusercontent.com/c0linhu1/credit-risk-scoring-model/main/credit_risk_dataset.csv"
CACHE = DATA_DIR / "credit_risk.csv"
CATEGORIES = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]

if CACHE.exists():
    brut, SOURCE = pd.read_csv(CACHE), "cache local"
else:
    try:
        from sklearn.datasets import fetch_openml
        brut, SOURCE = fetch_openml(data_id=43454, as_frame=True).frame, "OpenML (id 43454)"
    except Exception as erreur_openml:
        try:
            brut, SOURCE = pd.read_csv(URL_MIROIR), "miroir GitHub"
        except Exception as erreur_miroir:
            print("Réseau indisponible :", erreur_openml, "|", erreur_miroir)
            brut, SOURCE = pd.read_csv(io.StringIO(DONNEES_SECOURS)), "échantillon de secours (500 dossiers)"
    if "secours" not in SOURCE:
        brut.to_csv(CACHE, index=False)

brut[CATEGORIES] = brut[CATEGORIES].astype(str)          # fetch_openml renvoie des « category », le CSV des chaînes : on uniformise
brut["loan_status"] = brut["loan_status"].astype(int)
print(f"{len(brut)} dossiers chargés depuis : {SOURCE}")

In [ ]:
print("Dimensions :", brut.shape)
print(brut.dtypes.to_string(), "\n")
print("Valeurs manquantes :", brut.isna().sum()[brut.isna().sum() > 0].to_dict())
print(f"Taux de défaut : {brut['loan_status'].mean():.1%}  →  déséquilibre {1 - brut['loan_status'].mean():.0%} / {brut['loan_status'].mean():.0%}")
brut.head(3)

## 3. Nettoyage et feature engineering

Le journal des corrections : chaque ligne supprimée doit être justifiée par une **impossibilité physique**, pas par « ça m'arrange ». Les valeurs manquantes (`loan_int_rate`, `person_emp_length`) ne sont pas supprimées : elles seront **imputées dans le pipeline** (médiane apprise sur l'apprentissage seulement) — et XGBoost sait de toute façon les gérer.

In [ ]:
JOURNAL = []

def noter(action, avant, apres):
    JOURNAL.append({"étape": len(JOURNAL) + 1, "action": action, "lignes avant": avant, "lignes après": apres, "supprimées": avant - apres})
    print(f"{action:62s} {avant:6d} → {apres:6d}")

df = brut.copy()
n = len(df); df = df[df["person_age"] <= 100]
noter("Âge déclaré supérieur à 100 (erreur de saisie)", n, len(df))
n = len(df); df = df[df["person_emp_length"].isna() | (df["person_emp_length"] <= 60)]
noter("Ancienneté d'emploi supérieure à 60 ans (impossible)", n, len(df))
n = len(df); df = df[df["person_income"] < 2_000_000]
noter("Revenu annuel ≥ 2 M$ (hors cible de la banque en ligne)", n, len(df))
n = len(df); df = df.drop_duplicates()
noter("Dossiers en double", n, len(df))
df = df.reset_index(drop=True)
JOURNAL.append({"étape": len(JOURNAL) + 1, "action": "loan_int_rate et person_emp_length manquants : conservés, imputés dans le pipeline",
                "lignes avant": len(df), "lignes après": len(df), "supprimées": 0})

### Nouvelles variables

- `grade_num` : la note `A`→`G` transformée en nombre 1→7 (c'est une échelle **ordonnée** : un one-hot perdrait l'ordre).
- `age_premier_credit` : âge − longueur de l'historique de crédit = à quel âge le client a ouvert sa première ligne de crédit.
- `revenu_log` : le revenu en log (quelques très hauts revenus écrasent l'échelle).
- `ratio_verifie` : on recalcule mensualité / revenu pour vérifier `loan_percent_income` (une variable fournie n'est pas forcément juste).

**À toi (1)** · Crée `grade_num` : `A` → 1, `B` → 2, … `G` → 7.

<details><summary>Indice</summary>

`dict(zip("ABCDEFG", range(1, 8)))` fabrique le dictionnaire, puis `.map(...)`.
</details>

In [ ]:
# À toi : remplace la valeur provisoire par la bonne transformation
df["grade_num"] = 0        # provisoire
print(df.groupby("loan_grade")["grade_num"].first())

In [ ]:
verifier("Exercice 1 · grade_num va de 1 à 7 sans manquant", df["grade_num"].notna().all() and df["grade_num"].min() == 1 and df["grade_num"].max() == 7)
verifier("Exercice 1 · grade_num est très corrélé au taux d'intérêt", df["grade_num"].corr(df["loan_int_rate"]) > 0.8)

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Réinjecte l'implémentation de référence pour que la suite du notebook fonctionne.
try:
    _fait = df["grade_num"].min() == 1 and df["grade_num"].max() == 7
except Exception:
    _fait = False
if not _fait:
    df["grade_num"] = df["loan_grade"].map(dict(zip("ABCDEFG", range(1, 8))))

<details><summary>Solution</summary>

```python
df["grade_num"] = df["loan_grade"].map(dict(zip("ABCDEFG", range(1, 8))))
print(df.groupby("loan_grade")["grade_num"].first())
```
</details>

**À toi (2)** · Crée `age_premier_credit` = `person_age` − `cb_person_cred_hist_length`, et `revenu_log` = `np.log1p(person_income)`. Puis `ratio_verifie` = `loan_amnt / person_income` arrondi à 2 décimales : est-il égal à `loan_percent_income` ?

<details><summary>Indice</summary>

Trois affectations simples ; pour comparer, `(df["ratio_verifie"] - df["loan_percent_income"]).abs().max()`.
</details>

In [ ]:
# À toi
df["age_premier_credit"] = df["person_age"]          # provisoire
df["revenu_log"] = df["person_income"]                # provisoire
df["ratio_verifie"] = df["loan_percent_income"]       # provisoire
print("Écart max entre ratio recalculé et ratio fourni :", (df["ratio_verifie"] - df["loan_percent_income"]).abs().max())

In [ ]:
verifier("Exercice 2 · age_premier_credit plus petit que l'âge", (df["age_premier_credit"] < df["person_age"]).all())
verifier("Exercice 2 · revenu_log est bien en log (max < 20)", df["revenu_log"].max() < 20)
verifier("Exercice 2 · ratio_verifie ≈ loan_percent_income (écart max < 0,02)", (df["ratio_verifie"] - df["loan_percent_income"]).abs().max() < 0.02)

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Réinjecte l'implémentation de référence pour que la suite du notebook fonctionne.
try:
    _fait = df["revenu_log"].max() < 20 and (df["age_premier_credit"] < df["person_age"]).all()
except Exception:
    _fait = False
if not _fait:
    df["age_premier_credit"] = df["person_age"] - df["cb_person_cred_hist_length"]
    df["revenu_log"] = np.log1p(df["person_income"])
    df["ratio_verifie"] = (df["loan_amnt"] / df["person_income"]).round(2)

<details><summary>Solution</summary>

```python
df["age_premier_credit"] = df["person_age"] - df["cb_person_cred_hist_length"]
df["revenu_log"] = np.log1p(df["person_income"])
df["ratio_verifie"] = (df["loan_amnt"] / df["person_income"]).round(2)
print("Écart max entre ratio recalculé et ratio fourni :", (df["ratio_verifie"] - df["loan_percent_income"]).abs().max())
```
</details>

In [ ]:
df["defaut_anterieur"] = (df["cb_person_default_on_file"] == "Y").astype(int)
journal = pd.DataFrame(JOURNAL)
print(f"Dossiers conservés : {len(df)} sur {len(brut)} ({len(df) / len(brut):.1%})")
journal

## 4. Analyse exploratoire

Quatre indicateurs qui répondent à « qui fait défaut ? », puis les corrélations. Tout est calculé sur l'ensemble des dossiers (l'exploration ne coûte rien, c'est la modélisation qu'on sous-échantillonnera en mode rapide).

In [ ]:
taux_global = df["loan_status"].mean()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df.groupby("loan_grade")["loan_status"].mean().plot.bar(ax=axes[0], color="tab:red")
axes[0].axhline(taux_global, ls="--", color="gray"); axes[0].set_title("Taux de défaut par note (grade)"); axes[0].set_ylabel("taux de défaut")
df.groupby("loan_intent")["loan_status"].mean().sort_values().plot.barh(ax=axes[1], color="tab:orange")
axes[1].axvline(taux_global, ls="--", color="gray"); axes[1].set_title("Par objet du prêt")
plt.tight_layout(); plt.show()
print(f"Taux global : {taux_global:.1%}")

**Lecture** : la note interne est déjà un excellent prédicteur (de ~10 % de défauts en A à plus de 50 % en E-F-G) — le modèle devra faire mieux que « refuser à partir de D ». L'objet du prêt joue peu, sauf la consolidation de dettes et le médical, un peu plus risqués.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df.groupby("person_home_ownership")["loan_status"].mean().sort_values().plot.bar(ax=axes[0], color="tab:purple")
axes[0].axhline(taux_global, ls="--", color="gray"); axes[0].set_title("Par situation de logement"); axes[0].set_ylabel("taux de défaut")
df["tranche_revenu"] = pd.qcut(df["person_income"], 10, labels=[f"D{i}" for i in range(1, 11)])
df.groupby("tranche_revenu", observed=True)["loan_status"].mean().plot(marker="o", ax=axes[1], color="tab:green")
axes[1].axhline(taux_global, ls="--", color="gray"); axes[1].set_title("Par décile de revenu (D1 = 10 % les plus bas)"); axes[1].set_ylim(0)
plt.tight_layout(); plt.show()

**Lecture** : les locataires font deux fois plus défaut que les propriétaires ; le taux de défaut chute avec le revenu, surtout dans les trois premiers déciles. Mais le revenu seul ne suffit pas : c'est le **poids du prêt dans le revenu** qui compte.

In [ ]:
df["tranche_ratio"] = pd.cut(df["loan_percent_income"], [0, 0.1, 0.2, 0.3, 0.4, 0.5, 1], include_lowest=True)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df.groupby("tranche_ratio", observed=True)["loan_status"].mean().plot.bar(ax=axes[0], color="tab:blue")
axes[0].set_title("Par part du revenu consacrée au prêt"); axes[0].set_ylabel("taux de défaut"); axes[0].tick_params(axis="x", rotation=30)
variables_num = ["loan_status", "grade_num", "loan_int_rate", "loan_percent_income", "revenu_log", "loan_amnt", "person_emp_length", "cb_person_cred_hist_length", "defaut_anterieur"]
corr = df[variables_num].corr()
im = axes[1].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr))); axes[1].set_xticklabels(corr.columns, rotation=60, ha="right"); axes[1].set_yticks(range(len(corr))); axes[1].set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        axes[1].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=axes[1]); axes[1].set_title("Corrélations"); plt.tight_layout(); plt.show()

**Lecture** : au-delà de 30 % du revenu, plus d'un prêt sur deux n'est pas remboursé — c'est la variable la plus discriminante avec la note. Sur la matrice, la cible est corrélée à `loan_percent_income` (0,38), au taux et à la note (0,33), négativement au revenu ; `grade_num` et `loan_int_rate` sont redondants (0,93) — un modèle linéaire n'aimera pas, les arbres s'en moquent.

**À toi (3)** · Compare le taux de défaut selon `cb_person_default_on_file` (`Y` = un défaut déjà enregistré). Range les deux taux dans `taux_Y` et `taux_N`.

<details><summary>Indice</summary>

`df.groupby("cb_person_default_on_file")["loan_status"].mean()` puis `.loc["Y"]` et `.loc["N"]`.
</details>

In [ ]:
# À toi
taux_Y = None
taux_N = None
print("Défaut si historique de défaut :", taux_Y, "   sinon :", taux_N)

In [ ]:
verifier("Exercice 3 · taux_Y et taux_N sont des taux entre 0 et 1", taux_Y is not None and taux_N is not None and 0 < taux_N < taux_Y < 1)
verifier("Exercice 3 · un défaut antérieur fait plus que doubler le risque", lambda: taux_Y / taux_N > 1.5)

<details><summary>Solution</summary>

```python
par_historique = df.groupby("cb_person_default_on_file")["loan_status"].mean()
taux_Y, taux_N = par_historique.loc["Y"], par_historique.loc["N"]
print(f"Défaut si historique de défaut : {taux_Y:.1%}   sinon : {taux_N:.1%}")
```
</details>

**Ce qui oriente le choix du modèle** : des effets de seuil (ratio > 30 %, note ≥ D), des variables redondantes, deux catégorielles utiles (logement, objet) et 22 % de positifs. Une régression logistique servira de référence lisible ; les **arbres boostés** sont les favoris pour capturer les seuils et les interactions (revenu × ratio × note). Et quel que soit le modèle, le vrai travail sera le **seuil**.

In [ ]:
print("=== Rapport · section 4 (indicateurs) ===")
print(f"Dossiers : {len(df)}   |   taux de défaut : {taux_global:.1%}   |   manquants : taux d'intérêt {df['loan_int_rate'].isna().mean():.0%}, ancienneté {df['person_emp_length'].isna().mean():.0%}")
par_grade = df.groupby("loan_grade")["loan_status"].mean()
print(f"Défaut par note : A {par_grade['A']:.0%} → D {par_grade['D']:.0%} → G {par_grade['G']:.0%}   |   ratio > 30 % du revenu : {df.loc[df['loan_percent_income'] > 0.3, 'loan_status'].mean():.0%} de défauts")
print(f"Corrélation la plus forte avec la cible : {corr['loan_status'].drop('loan_status').abs().idxmax()} ({corr['loan_status'].drop('loan_status').abs().max():.2f})")

## 5. Modèles candidats

**Protocole** : découpage **stratifié** 80 / 20 (même taux de défaut des deux côtés), validation croisée stratifiée à 5 plis sur l'apprentissage, test intouché jusqu'à la fin. En `MODE_RAPIDE`, on sous-échantillonne 2 000 dossiers (stratifiés) pour la modélisation. Chaque modèle est un `Pipeline` : imputation + standardisation des numériques, one-hot des catégories, modèle.

Métriques : **ROC-AUC** (validation croisée et test), **PR-AUC**, et au seuil 0,5 le **rappel** (part des défauts détectés), la précision et le F1 — pour voir tout de suite pourquoi le seuil 0,5 est un mauvais choix.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score, f1_score

VARS_NUM = ["person_age", "revenu_log", "person_emp_length", "loan_amnt", "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length", "grade_num", "age_premier_credit", "defaut_anterieur"]
VARS_CAT = ["person_home_ownership", "loan_intent"]
CIBLE = "loan_status"

modelisation = train_test_split(df, train_size=2000, stratify=df[CIBLE], random_state=SEED)[0] if MODE_RAPIDE and len(df) > 2000 else df
X, y = modelisation[VARS_NUM + VARS_CAT], modelisation[CIBLE]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"Apprentissage : {len(X_train)} dossiers ({y_train.mean():.1%} de défauts) · test : {len(X_test)} ({y_test.mean():.1%})")

In [ ]:
def faire_preprocesseur():
    num = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    return ColumnTransformer([("num", num, VARS_NUM), ("cat", cat, VARS_CAT)])

TABLEAU = []

def evaluer(nom, modele, seuil=0.5):
    """CV stratifiée (ROC-AUC) + mesures sur le test au seuil donné. `modele` = un estimateur, mis dans un Pipeline avec le préprocesseur."""
    debut = time.time()
    pipe = modele if hasattr(modele, "steps") else Pipeline([("prep", faire_preprocesseur()), ("modele", modele)])
    auc_cv = cross_val_score(pipe, X_train, y_train, cv=CV, scoring="roc_auc").mean()
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= seuil).astype(int)
    ligne = {"modèle": nom, "ROC-AUC CV": auc_cv, "ROC-AUC test": roc_auc_score(y_test, proba), "PR-AUC test": average_precision_score(y_test, proba),
             "rappel": recall_score(y_test, pred), "précision": precision_score(y_test, pred, zero_division=0), "F1": f1_score(y_test, pred), "temps (s)": time.time() - debut}
    TABLEAU.append(ligne)
    print(f"{nom:34s} AUC CV {auc_cv:.3f} · AUC test {ligne['ROC-AUC test']:.3f} · PR-AUC {ligne['PR-AUC test']:.3f} · rappel {ligne['rappel']:.2f} · précision {ligne['précision']:.2f} · {ligne['temps (s)']:.1f}s")
    return pipe

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

N_ARBRES = 200 if MODE_RAPIDE else 500
modele_baseline = evaluer("Baseline (tirage au sort 22 %)", DummyClassifier(strategy="stratified", random_state=SEED))
modele_logreg = evaluer("Régression logistique", LogisticRegression(max_iter=2000))
modele_rf = evaluer("Random Forest", RandomForestClassifier(n_estimators=N_ARBRES, min_samples_leaf=3, random_state=SEED, n_jobs=-1))
modele_xgb = evaluer("XGBoost (défaut)", XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=4, random_state=SEED, n_jobs=-1))
tableau = pd.DataFrame(TABLEAU).set_index("modèle").round(3)
tableau

**Lecture** : tous les modèles classent bien (AUC > 0,85) mais regarde la colonne **rappel** au seuil 0,5 : la régression logistique laisse passer une bonne partie des défauts. Avec 22 % de positifs, un modèle « prudent » prédit 0 dès qu'il hésite.

### Le déséquilibre : `class_weight` ou SMOTE ?

Deux remèdes classiques, comparés sur la régression logistique :
- **`class_weight="balanced"`** : chaque défaut pèse ~3,5 fois plus dans la fonction de coût — rien à changer dans les données.
- **SMOTE** (`imbalanced-learn`) : on fabrique des défauts synthétiques par interpolation entre voisins, **dans l'apprentissage seulement** (le `Pipeline` d'imblearn s'en charge à chaque pli — jamais sur le test).

In [ ]:
from imblearn.pipeline import Pipeline as PipelineImb
from imblearn.over_sampling import SMOTE

modele_logreg_pond = evaluer("Rég. logistique + class_weight", LogisticRegression(max_iter=2000, class_weight="balanced"))
modele_logreg_smote = evaluer("Rég. logistique + SMOTE",
                              PipelineImb([("prep", faire_preprocesseur()), ("smote", SMOTE(random_state=SEED)), ("modele", LogisticRegression(max_iter=2000))]))
tableau = pd.DataFrame(TABLEAU).set_index("modèle").round(3)
tableau.loc[[m for m in tableau.index if "logistique" in m], ["ROC-AUC CV", "PR-AUC test", "rappel", "précision", "F1"]]

**Lecture** : les deux remèdes font grimper le rappel (on détecte bien plus de défauts) au prix de la précision, et l'AUC ne bouge presque pas — logique, l'AUC ne dépend pas du seuil. Rééquilibrer revient en fait à **déplacer le seuil** ; on le fera proprement à la section 6, avec un coût métier.

**À toi (4)** · Même comparaison pour XGBoost : son paramètre s'appelle `scale_pos_weight` (poids des positifs ; mets le ratio négatifs / positifs de l'apprentissage). Évalue `modele_xgb_pond` et range le ratio dans `RATIO`.

<details><summary>Indice</summary>

`RATIO = (y_train == 0).sum() / (y_train == 1).sum()` puis `XGBClassifier(..., scale_pos_weight=RATIO)`.
</details>

In [ ]:
# À toi
RATIO = None
modele_xgb_pond = None

In [ ]:
verifier("Exercice 4 · RATIO ≈ négatifs / positifs (entre 3 et 4)", RATIO is not None and 3 < RATIO < 4.5)
verifier("Exercice 4 · le rappel de XGBoost pondéré dépasse celui du XGBoost par défaut",
         lambda: [l for l in TABLEAU if l["modèle"] == "XGBoost + scale_pos_weight"][0]["rappel"] > tableau.loc["XGBoost (défaut)", "rappel"])

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Réinjecte l'implémentation de référence pour que la suite du notebook fonctionne.
try:
    _fait = any(ligne["modèle"] == "XGBoost + scale_pos_weight" for ligne in TABLEAU)
except Exception:
    _fait = False
if not _fait:
    RATIO = (y_train == 0).sum() / (y_train == 1).sum()
    modele_xgb_pond = evaluer("XGBoost + scale_pos_weight",
                              XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=4, scale_pos_weight=RATIO,
                                            random_state=SEED, n_jobs=-1))

<details><summary>Solution</summary>

```python
RATIO = (y_train == 0).sum() / (y_train == 1).sum()
modele_xgb_pond = evaluer("XGBoost + scale_pos_weight", XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=4, scale_pos_weight=RATIO, random_state=SEED, n_jobs=-1))
```
</details>

In [ ]:
tableau = pd.DataFrame(TABLEAU).set_index("modèle").round(3)
print("=== Rapport · section 5 (modèles candidats) ===")
print(tableau[["ROC-AUC CV", "ROC-AUC test", "PR-AUC test", "rappel", "précision", "temps (s)"]].to_string())
meilleur = tableau["ROC-AUC CV"].idxmax()
print(f"\nMeilleur candidat en CV : {meilleur} (AUC {tableau.loc[meilleur, 'ROC-AUC CV']:.3f})")

## 6. Tuning : XGBoost + Optuna, puis le seuil de décision

**Optuna** remplace le tirage aléatoire de `RandomizedSearchCV` par une recherche **bayésienne** : chaque essai est choisi en fonction des précédents (algorithme TPE), les zones prometteuses sont explorées davantage. On écrit une fonction `objectif(trial)` qui construit un modèle avec les paramètres proposés et renvoie l'AUC en validation croisée ; Optuna la maximise. En `MODE_RAPIDE` : 20 essais × 3 plis ; complet : 60 essais × 5 plis.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_ESSAIS, N_PLIS = (20, 3) if MODE_RAPIDE else (60, 5)
CV_TUNING = StratifiedKFold(n_splits=N_PLIS, shuffle=True, random_state=SEED)

def objectif(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 5.0),
    }
    pipe = Pipeline([("prep", faire_preprocesseur()), ("modele", XGBClassifier(**params, random_state=SEED, n_jobs=-1))])
    return cross_val_score(pipe, X_train, y_train, cv=CV_TUNING, scoring="roc_auc").mean()

etude = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
debut = time.time()
etude.optimize(objectif, n_trials=N_ESSAIS)
print(f"{N_ESSAIS} essais en {time.time() - debut:.0f} s · meilleur AUC CV : {etude.best_value:.4f}")
MEILLEURS_PARAMS = {k: (round(v, 3) if isinstance(v, float) else v) for k, v in etude.best_params.items()}
print("Meilleurs paramètres :", MEILLEURS_PARAMS)

In [ ]:
valeurs = [t.value for t in etude.trials]
plt.plot(valeurs, "o", alpha=0.5, label="AUC de l'essai")
plt.plot(np.maximum.accumulate(valeurs), "-", color="tab:red", label="meilleur AUC jusqu'ici")
plt.xlabel("numéro d'essai"); plt.ylabel("ROC-AUC (CV)"); plt.legend(); plt.title("Optuna : la recherche converge")
plt.show()
modele_xgb_tune = evaluer("XGBoost (Optuna)", XGBClassifier(**etude.best_params, random_state=SEED, n_jobs=-1))
tableau = pd.DataFrame(TABLEAU).set_index("modèle").round(3)
tableau[["ROC-AUC CV", "ROC-AUC test", "PR-AUC test", "rappel", "précision", "F1"]]

### Le seuil de décision : un choix métier, pas statistique

Le modèle donne une probabilité de défaut ; la banque doit trancher. Avec **FN = 10 × FP** (un défaut manqué coûte dix fois un bon client refusé), on calcule le coût total pour chaque seuil possible — sur des **probabilités obtenues en validation croisée** sur l'apprentissage (`cross_val_predict`), jamais sur le test, sinon on ajusterait le seuil sur les réponses.

In [ ]:
from sklearn.metrics import confusion_matrix

COUT_FN, COUT_FP = 10, 1
proba_cv = cross_val_predict(modele_xgb_tune, X_train, y_train, cv=CV, method="predict_proba")[:, 1]

def cout_total(seuil, proba=proba_cv, y=y_train.values, cout_fn=COUT_FN, cout_fp=COUT_FP):
    pred = proba >= seuil
    fn, fp = ((~pred) & (y == 1)).sum(), (pred & (y == 0)).sum()
    return cout_fn * fn + cout_fp * fp

SEUILS = np.linspace(0.05, 0.95, 91)
couts = np.array([cout_total(s) for s in SEUILS])
SEUIL = float(SEUILS[couts.argmin()])
plt.plot(SEUILS, couts); plt.axvline(0.5, ls="--", color="gray", label="seuil naïf 0,5"); plt.axvline(SEUIL, color="tab:red", label=f"seuil optimal {SEUIL:.2f}")
plt.xlabel("seuil de probabilité (au-dessus → refus)"); plt.ylabel("coût total (unités de FP)"); plt.legend(); plt.title("Coût métier en fonction du seuil (validation croisée)")
plt.show()
print(f"Coût au seuil 0,5 : {cout_total(0.5)}   →   au seuil {SEUIL:.2f} : {couts.min()}   ({1 - couts.min() / cout_total(0.5):.0%} de moins)")

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

proba_test = modele_xgb_tune.predict_proba(X_test)[:, 1]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax, seuil in zip(axes[:2], [0.5, SEUIL]):
    cm = confusion_matrix(y_test, proba_test >= seuil)
    ax.imshow(cm, cmap="Blues"); ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(["accordé", "refusé"]); ax.set_yticklabels(["remboursé", "défaut"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13, color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_title(f"Seuil {seuil:.2f} · coût {cout_total(seuil, proba_test, y_test.values)}")
RocCurveDisplay.from_predictions(y_test, proba_test, ax=axes[2], name="XGBoost Optuna"); axes[2].set_title("Courbe ROC (test)")
PrecisionRecallDisplay.from_predictions(y_test, proba_test, ax=axes[3], name="XGBoost Optuna"); axes[3].set_title("Courbe précision-rappel (test)")
plt.tight_layout(); plt.show()

**Lecture** : au seuil optimal, on refuse plus de bons clients (case « remboursé / refusé ») mais on laisse passer beaucoup moins de défauts — et le coût total baisse. La courbe ROC dit « le modèle classe bien », la courbe PR dit « à quel prix en précision on gagne du rappel » ; le seuil est le curseur sur cette courbe.

**À toi (5)** · Écris `seuil_pour(cout_fn)` qui renvoie le seuil optimal pour un coût de faux négatif donné (FP restant à 1), en réutilisant `cout_total`. Vérifie l'intuition : plus le défaut coûte cher, plus le seuil doit être **bas**.

<details><summary>Indice</summary>

Même calcul que ci-dessus : `couts = [cout_total(s, cout_fn=cout_fn) for s in SEUILS]` puis `SEUILS[np.argmin(couts)]`.
</details>

In [ ]:
# À toi
def seuil_pour(cout_fn):
    return 0.5                        # provisoire

for c in [1, 2, 5, 10, 20]:
    print(f"FN = {c:2d} × FP  →  seuil optimal {seuil_pour(c):.2f}")

In [ ]:
verifier("Exercice 5 · seuil_pour(10) retrouve le seuil calculé plus haut", proche(seuil_pour(10), SEUIL, 0.01))
verifier("Exercice 5 · le seuil baisse quand le défaut coûte plus cher", seuil_pour(1) > seuil_pour(5) >= seuil_pour(20))

<details><summary>Solution</summary>

```python
def seuil_pour(cout_fn):
    couts = [cout_total(s, cout_fn=cout_fn) for s in SEUILS]
    return float(SEUILS[np.argmin(couts)])

for c in [1, 2, 5, 10, 20]:
    print(f"FN = {c:2d} × FP  →  seuil optimal {seuil_pour(c):.2f}")
```
</details>

In [ ]:
print("=== Rapport · section 6 (tuning et seuil) ===")
avant, apres = tableau.loc["XGBoost (défaut)"], tableau.loc["XGBoost (Optuna)"]
print(f"ROC-AUC CV : {avant['ROC-AUC CV']:.3f} → {apres['ROC-AUC CV']:.3f}   |   ROC-AUC test : {avant['ROC-AUC test']:.3f} → {apres['ROC-AUC test']:.3f}   |   PR-AUC test : {avant['PR-AUC test']:.3f} → {apres['PR-AUC test']:.3f}   ({N_ESSAIS} essais Optuna)")
print(f"Seuil retenu (FN = {COUT_FN} × FP) : {SEUIL:.2f}   |   coût test au seuil 0,5 : {cout_total(0.5, proba_test, y_test.values)} → au seuil {SEUIL:.2f} : {cout_total(SEUIL, proba_test, y_test.values)}")
print(f"Rappel test : {recall_score(y_test, proba_test >= 0.5):.2f} → {recall_score(y_test, proba_test >= SEUIL):.2f}   |   précision : {precision_score(y_test, proba_test >= 0.5):.2f} → {precision_score(y_test, proba_test >= SEUIL):.2f}")
print("Paramètres retenus :", MEILLEURS_PARAMS)

## 7. Interprétation

Deux niveaux : **global** (quelles variables pilotent le score, pour la direction des risques) et **local** (pourquoi *ce* dossier est refusé, pour l'analyste et le client). Les valeurs de Shapley (`shap`) répondent aux deux avec la même logique : la somme des contributions = score du dossier − score moyen. Pour XGBoost, elles sont exprimées en **log-odds** (le score avant la transformation en probabilité).

In [ ]:
import shap

prep_final = modele_xgb_tune.named_steps["prep"]
noms_variables = [n.split("__", 1)[1] for n in prep_final.get_feature_names_out()]
X_test_p = pd.DataFrame(prep_final.transform(X_test), columns=noms_variables, index=X_test.index)
explainer = shap.Explainer(modele_xgb_tune.named_steps["modele"], feature_names=noms_variables)
shap_values = explainer(X_test_p)

shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title("Importance globale : chaque point = un dossier ; rouge = valeur élevée"); plt.tight_layout(); plt.show()

**Lecture** : le poids du prêt dans le revenu, la note et le revenu dominent ; un ratio élevé (rouge) pousse fortement vers le défaut, un revenu élevé protège. Le logement en location (`person_home_ownership_RENT`) est le facteur catégoriel le plus visible. Aucune variable ne fait tout : c'est bien une combinaison.

### La fiche « décision expliquée »

Comme dans la soutenance P7, chaque dossier a droit à sa fiche : la probabilité face au seuil, la décision, et les 6 facteurs qui ont le plus pesé — avec un graphique lisible par un non-technicien.

In [ ]:
def expliquer(i):
    """Fiche du i-ème dossier du test : probabilité, décision, contributions SHAP triées."""
    contributions = pd.Series(shap_values.values[i], index=noms_variables)
    contributions = contributions.reindex(contributions.abs().sort_values(ascending=False).index)
    proba = float(proba_test[i])
    return {"position": i, "proba": proba, "seuil": SEUIL, "decision": "REFUS" if proba >= SEUIL else "ACCORD",
            "reel": "défaut" if y_test.iloc[i] == 1 else "remboursé", "contributions": contributions.head(6), "dossier": X_test.iloc[i]}

def dessiner_fiche(fiche):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), width_ratios=[0.8, 1.4, 1])
    axes[0].barh(["risque"], [fiche["proba"]], color="tab:red" if fiche["decision"] == "REFUS" else "tab:green")
    axes[0].axvline(fiche["seuil"], ls="--", color="black"); axes[0].set_xlim(0, 1); axes[0].set_title(f"Probabilité de défaut {fiche['proba']:.0%} · seuil {fiche['seuil']:.0%}")
    c = fiche["contributions"][::-1]
    axes[1].barh(c.index, c.values, color=["tab:red" if v > 0 else "tab:green" for v in c.values]); axes[1].axvline(0, color="black")
    axes[1].set_title("Ce qui pèse (rouge = vers le refus, vert = vers l'accord)")
    axes[2].hist(X_test["loan_percent_income"].dropna(), bins=30, color="lightgray"); axes[2].axvline(fiche["dossier"]["loan_percent_income"], color="tab:red", lw=2)
    axes[2].set_title("Part du revenu consacrée au prêt : le client vs les autres")
    fig.suptitle(f"Dossier n°{fiche['position']} · décision : {fiche['decision']}   (réalité : {fiche['reel']})", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

refus = int(np.argmax(proba_test >= SEUIL))               # premier dossier refusé du test
dessiner_fiche(expliquer(refus))

In [ ]:
shap.plots.waterfall(shap_values[refus], max_display=10, show=False)
plt.title("Le même dossier vu par SHAP (échelle log-odds)"); plt.tight_layout(); plt.show()

**À toi (6)** · Explique un dossier **accordé** : trouve la position `accord` du premier dossier du test dont la probabilité est sous le seuil, construis `ma_fiche = expliquer(accord)` et dessine-la. Quel est le facteur qui a le plus joué en sa faveur ?

<details><summary>Indice</summary>

`int(np.argmax(proba_test < SEUIL))` ; le facteur le plus favorable est la contribution la plus négative : `ma_fiche["contributions"].idxmin()`.
</details>

In [ ]:
# À toi
accord = None
ma_fiche = None

In [ ]:
verifier("Exercice 6 · ma_fiche est une fiche d'un dossier accordé", ma_fiche is not None and ma_fiche["decision"] == "ACCORD" and ma_fiche["proba"] < SEUIL)
verifier("Exercice 6 · la fiche contient 6 contributions", lambda: len(ma_fiche["contributions"]) == 6)

<details><summary>Solution</summary>

```python
accord = int(np.argmax(proba_test < SEUIL))
ma_fiche = expliquer(accord)
dessiner_fiche(ma_fiche)
print("Facteur le plus favorable :", ma_fiche["contributions"].idxmin())
```
</details>

In [ ]:
pred_finale = (proba_test >= SEUIL).astype(int)
erreurs = pd.DataFrame({"note": X_test["grade_num"].values, "ratio": X_test["loan_percent_income"].values, "vrai": y_test.values, "pred": pred_finale})
manques = erreurs[(erreurs["vrai"] == 1) & (erreurs["pred"] == 0)]          # défauts non détectés (FN)
detectes = erreurs[(erreurs["vrai"] == 1) & (erreurs["pred"] == 1)]
print(f"Défauts non détectés : {len(manques)} sur {int(erreurs['vrai'].sum())}")
print(f"Note moyenne : défauts manqués {manques['note'].mean():.2f} vs détectés {detectes['note'].mean():.2f}   |   ratio moyen : {manques['ratio'].mean():.2f} vs {detectes['ratio'].mean():.2f}")

**Trois enseignements**
1. Le **ratio prêt / revenu** et la **note** expliquent l'essentiel du score ; le revenu et le logement viennent ensuite. Le modèle a retrouvé, et affiné, les règles de bon sens des analystes.
2. Les défauts que le modèle **manque** sont des dossiers « propres » : bonne note, ratio faible — ils ressemblent à des bons clients. Aucune variable disponible ne permet de les distinguer ; il faudrait d'autres données (comportement bancaire, incidents récents).
3. Le **seuil** pèse plus que le tuning : Optuna gagne quelques millièmes d'AUC, le seuil optimisé fait baisser le coût de plusieurs dizaines de pourcents.

In [ ]:
print("=== Rapport · section 7 (interprétation) ===")
top3 = pd.Series(np.abs(shap_values.values).mean(axis=0), index=noms_variables).sort_values(ascending=False).head(3)
print("Top 3 variables (|SHAP| moyen) :", {k: round(v, 3) for k, v in top3.items()})
print(f"Défauts non détectés au seuil {SEUIL:.2f} : {len(manques)} / {int(erreurs['vrai'].sum())}  (note moyenne {manques['note'].mean():.2f} vs {detectes['note'].mean():.2f} pour les détectés)")
print(f"Dossier refusé illustré : n°{refus}, probabilité {proba_test[refus]:.0%}, facteur principal : {expliquer(refus)['contributions'].index[0]}")

## 8. Conclusion et recommandation

**Réponse à la question métier** : oui, on peut automatiser une première décision. Un XGBoost réglé par Optuna classe les dossiers avec un ROC-AUC ≈ 0,93-0,95, et surtout un **seuil de décision abaissé** (≈ 0,2-0,3 au lieu de 0,5) réduit le coût métier de façon nette quand un défaut coûte dix fois un refus. Chaque décision est accompagnée d'une fiche qui nomme les facteurs déterminants — condition pour le régulateur comme pour le client.

**Chiffre clé** : à recopier depuis la cellule Rapport de la section 6 (coût test au seuil 0,5 → au seuil optimal).

**Limites** : le rapport de coût 10 : 1 est une hypothèse (à négocier avec la direction financière : le coût réel dépend du montant et de la marge de chaque prêt) ; le dataset ne dit ni quand ni combien a été perdu ; les variables sont déclaratives ; aucune vérification d'équité entre groupes (logement, âge) n'a été faite — indispensable avant une mise en production.

**Avec plus de temps** : un coût **par dossier** (montant × probabilité) à la place d'un coût fixe, une calibration des probabilités (`CalibratedClassifierCV`), un audit d'équité (taux de refus par groupe), et un suivi dans le temps (dérive du score).

In [ ]:
MA_SYNTHESE = """
(Remplace ce texte par 5 lignes : ta réponse à la question métier, ton chiffre clé, ta recommandation.)
"""
print(MA_SYNTHESE.strip())

## 9. Pour aller plus loin

- Le dataset : [Credit Risk Dataset sur OpenML](https://www.openml.org/d/43454) · [sur Kaggle](https://www.kaggle.com/datasets/laotse/credit-risk-dataset) (avec des dizaines de notebooks publics à comparer au tien).
- Choisir un seuil avec scikit-learn directement : [`TunedThresholdClassifierCV`](https://scikit-learn.org/stable/modules/classification_threshold.html) (coût métier en une ligne).
- Tout sur le déséquilibre : [documentation imbalanced-learn](https://imbalanced-learn.org/stable/) — et pourquoi SMOTE n'est pas toujours une bonne idée avec les arbres boostés.
- [Optuna : tutoriel officiel](https://optuna.readthedocs.io/en/stable/tutorial/index.html) (pruning, visualisations, recherche multi-objectifs).
- [Documentation SHAP](https://shap.readthedocs.io/en/latest/) : waterfall, force plot, interactions.
- Mettre la fiche en ligne : une appli **Streamlit** qui prend un dossier et affiche `dessiner_fiche()` — c'est exactement le dashboard de la soutenance P7.
- Les briques Le Wagon d'origine (en anglais) : `data-challenges-en/05-ML/03-Performance-metrics` et `05-ML/05-Model-Tuning`.